In [20]:
!apt-get -y install -qq ffmpeg
!pip install -q gradio pydub speechrecognition scikit-learn gTTS

In [21]:
import re, json, textwrap
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --------- Política de seguridad (triage) ---------
EMERGENCY_PATTERNS = [
    r"dolor (de )?pecho (intenso|fuerte)",
    r"dificultad (severa )?para respirar|ahogo|no puedo respirar",
    r"cara ca[ií]da|debilidad en un lado|habla arrastrada|ictus|acv",
    r"p[eé]rdida de conocimiento|desmayo prolongado",
    r"sangrado (abundante|que no cede)",
    r"reacci[oó]n al[eé]rgica (grave|anafilaxia)|hinchaz[oó]n de garganta",
    r"convulsiones (prolongadas|por primera vez)",
    r"dolor abdominal intenso y persistente|embarazo y sangrado"
]

# Compilamos con flags para que sea más robusto
EMERGENCY_REGEX = [re.compile(p, flags=re.IGNORECASE) for p in EMERGENCY_PATTERNS]

def needs_emergency(text: str) -> bool:
    t = (text or "").lower()
    return any(r.search(t) for r in EMERGENCY_REGEX)  # ← corregido (sin paréntesis extra)

# --------- Base de conocimiento breve (FAQ) ---------
DOCS = [
    {"q":"síntomas de gripe",
     "a":"La gripe suele causar fiebre, dolor muscular, dolor de garganta, tos seca, dolor de cabeza y cansancio. Reposo, hidratación y analgésicos habituales pueden aliviar. Si hay dificultad para respirar, fiebre alta persistente o empeoramiento, urgencias."},
    {"q":"diferencia resfriado y gripe",
     "a":"Resfriado: inicio gradual, estornudos, congestión y tos leve, rara vez fiebre alta. Gripe: inicio brusco con fiebre, dolor muscular intenso y gran cansancio."},
    {"q":"síntomas covid",
     "a":"COVID-19 puede causar fiebre, tos, dolor de garganta, congestión, pérdida de olfato/sabor, dolores musculares y, a veces, dificultad respiratoria. Prueba diagnóstica y medidas de aislamiento según normativa local."},
    {"q":"fiebre en adultos",
     "a":"Fiebre es ≥38°C. Vigila hidratación, reposa y controla temperatura. Acude a urgencias si supera ~40°C, dura >3 días, hay confusión, rigidez de cuello, erupción extensa o dificultad para respirar."},
    {"q":"fiebre en niños",
     "a":"En pediatría importa la edad y el estado general. Urgencias si <3 meses con ≥38°C, somnolencia marcada, irritabilidad inconsolable, rigidez de cuello, respiración dificultosa o deshidratación."},
    {"q":"dolor de garganta que hago",
     "a":"Hidratación, analgésicos habituales y gárgaras con agua tibia y sal pueden aliviar. Si hay fiebre alta persistente, exudados, o dura >1 semana, consulta médica."},
    {"q":"gastroenteritis cuidados",
     "a":"Suele mejorar en 24–72 h. Hidrátate con sorbos frecuentes, dieta blanda y reposo. Urgencias si hay signos de deshidratación, sangre en heces, fiebre alta persistente o dolor abdominal intenso."},
    {"q":"medicamentos y dosis",
     "a":"No puedo indicar dosis personalizadas. Para medicamentos (p. ej., analgésicos o antibióticos) consulta a un profesional con tu historial y peso. Evita automedicarte y no uses antibióticos sin receta."},
    {"q":"prevención general",
     "a":"Lávate las manos, ventila espacios, cubre tos/estornudos, mantén al día tus vacunas, descansa y mantén una dieta equilibrada."},
]

corpus = [d["q"] for d in DOCS]
vectorizer = TfidfVectorizer()
corpus_vec = vectorizer.fit_transform(corpus)

DISCLAIMER = (
    "Este bot ofrece información general y no reemplaza la valoración de un profesional. "
    "Si tus síntomas son graves o empeoran, busca atención médica."
)

def retrieve_answer(user_text: str, k=2, threshold=0.15):
    qv = vectorizer.transform([ (user_text or "").lower() ])
    sims = cosine_similarity(qv, corpus_vec).ravel()
    idxs = sims.argsort()[::-1][:k]
    if sims[idxs[0]] < threshold:
        return None, 0.0
    picked = [DOCS[i]["a"] for i in idxs if sims[i] >= threshold]
    return " ".join(picked[:2]), float(sims[idxs[0]])

def medical_bot(user_text: str):
    if needs_emergency(user_text):
        return (
            "**Podría ser una urgencia.** Llama al servicio de emergencias "
            "o acude a un centro sanitario de inmediato.",
            True
        )

    ans, score = retrieve_answer(user_text)
    if ans:
        return f"{ans}\n\n_{DISCLAIMER}_", False

    # Fallback seguro si no encuentra coincidencias
    return (
        "No tengo suficiente contexto para responder con fiabilidad. "
        "Describe síntomas, duración y antecedentes relevantes. " + DISCLAIMER,
        False
    )


In [22]:
for q in [
    "Tengo fiebre y dolor muscular",
    "¿Cuál es la diferencia entre resfriado y gripe?",
    "Me duele el pecho y me falta el aire"
]:
    print("Q:", q)
    a, urg = medical_bot(q)
    print("A:", a, "| URGENCIA:", urg)
    print("-"*60)


Q: Tengo fiebre y dolor muscular
A: Hidratación, analgésicos habituales y gárgaras con agua tibia y sal pueden aliviar. Si hay fiebre alta persistente, exudados, o dura >1 semana, consulta médica. Fiebre es ≥38°C. Vigila hidratación, reposa y controla temperatura. Acude a urgencias si supera ~40°C, dura >3 días, hay confusión, rigidez de cuello, erupción extensa o dificultad para respirar.

_Este bot ofrece información general y no reemplaza la valoración de un profesional. Si tus síntomas son graves o empeoran, busca atención médica._ | URGENCIA: False
------------------------------------------------------------
Q: ¿Cuál es la diferencia entre resfriado y gripe?
A: Resfriado: inicio gradual, estornudos, congestión y tos leve, rara vez fiebre alta. Gripe: inicio brusco con fiebre, dolor muscular intenso y gran cansancio. La gripe suele causar fiebre, dolor muscular, dolor de garganta, tos seca, dolor de cabeza y cansancio. Reposo, hidratación y analgésicos habituales pueden aliviar. Si

In [23]:
!pip install -q transformers torch scikit-learn

In [15]:
CORPUS = {
"gripe_vs_resfriado": """
La gripe es una infección respiratoria por virus influenza con inicio brusco y fiebre alta,
dolor muscular intenso, cefalea y gran cansancio. El resfriado común suele tener inicio
gradual, congestión nasal, estornudos y dolor de garganta leve, con fiebre ausente o baja.
""",
"covid_sintomas": """
COVID-19 puede causar fiebre, tos, dolor de garganta, congestión nasal, pérdida de olfato
o gusto, dolores musculares, dolor de cabeza y, en algunos casos, dificultad respiratoria.
""",
"fiebre_adultos": """
En adultos se considera fiebre a partir de 38°C. Medidas: hidratación, reposo y control
de temperatura. Urgencias si supera ~40°C, dura >3 días, o aparecen confusión, rigidez
de cuello, erupción extensa o dificultad respiratoria importante.
""",
"fiebre_ninos": """
En niños, la valoración depende de la edad y del estado general. Es urgencia si un lactante
<3 meses tiene ≥38°C o si hay decaimiento extremo, irritabilidad inconsolable, rigidez de
cuello, respiración dificultosa o signos de deshidratación.
""",
"gastroenteritis": """
La gastroenteritis aguda suele mejorar en 24–72 h. Medidas: hidratación con sorbos
frecuentes, dieta blanda y reposo. Urgencias si hay sangre en heces, fiebre alta persistente,
dolor abdominal intenso o signos de deshidratación (orina escasa, sed intensa, mareo).
""",
"garganta": """
El dolor de garganta suele ser viral. Medidas: hidratación, analgésicos habituales si no hay
contraindicaciones y gárgaras con agua tibia y sal. Si hay fiebre alta persistente, exudados,
dificultad respiratoria o dura >1 semana, consulta médica.
""",
"prevencion": """
Prevención de infecciones respiratorias: lavado de manos, ventilar espacios, cubrir tos y
estornudos, evitar contacto con personas enfermas y mantener vacunas al día.
"""
}



In [16]:
import re
EMERGENCY_PATTERNS = [
    r"dolor (de )?pecho (intenso|fuerte)",
    r"dificultad (severa )?para respirar|ahogo|no puedo respirar",
    r"cara ca[ií]da|debilidad en un lado|habla arrastrada|ictus|acv",
    r"p[eé]rdida de conocimiento|desmayo prolongado",
    r"sangrado (abundante|que no cede)",
    r"reacci[oó]n al[eé]rgica (grave|anafilaxia)|hinchaz[oó]n de garganta",
    r"convulsiones (prolongadas|por primera vez)",
    r"dolor abdominal intenso y persistente|embarazo y sangrado"
]
EMERGENCY_REGEX = [re.compile(p, re.IGNORECASE) for p in EMERGENCY_PATTERNS]

DISCLAIMER = ("Este bot ofrece información general y no sustituye la evaluación de un profesional. "
              "Si los síntomas son graves o empeoran, busca atención médica.")

def needs_emergency(text:str)->bool:
    t = (text or "").lower()
    return any(r.search(t) for r in EMERGENCY_REGEX)



In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

doc_ids = list(CORPUS.keys())
docs = [CORPUS[i] for i in doc_ids]
vectorizer = TfidfVectorizer().fit(docs)
doc_matrix = vectorizer.transform(docs)

def retrieve_context(question:str, k:int=2):
    qv = vectorizer.transform([question])
    sims = cosine_similarity(qv, doc_matrix).ravel()
    top = sims.argsort()[::-1][:k]
    return [(doc_ids[i], docs[i], float(sims[i])) for i in top]


In [18]:
from transformers import pipeline
import torch
device = 0 if torch.cuda.is_available() else -1

# Modelo QA español (SQuAD-es). Fiable para preguntas-respuestas.
qa = pipeline(
    "question-answering",
    model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",
    tokenizer="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",
    device=device
)

def answer(question:str, k_ctx:int=2, qa_threshold:float=0.18)->str:
    if needs_emergency(question):
        return ("Podría tratarse de una urgencia. Acude de inmediato a urgencias "
                "o llama al número de emergencias. " + DISCLAIMER)

    ctxs = retrieve_context(question, k=k_ctx)

    best_ans, best_score = None, 0.0
    for _, ctx, _ in ctxs:
        try:
            out = qa(question=question, context=ctx)
            if out["score"] > best_score:
                best_ans, best_score = out["answer"], float(out["score"])
        except Exception:
            continue

    if best_ans and best_score >= qa_threshold:
        return f"{best_ans}\n\n_{DISCLAIMER}_"

    # Fallback seguro (sin generar): devolvemos la parte más relevante del mejor contexto
    if ctxs:
        best_ctx = ctxs[0][1].strip()
        # devolvemos 2-3 frases del contexto como guía general
        snippet = " ".join(best_ctx.split(".")[:3]).strip()
        return f"{snippet}.\n\n_{DISCLAIMER}_"

    return ("No tengo suficiente información para responder con fiabilidad. "
            "Aporta síntomas, duración y antecedentes relevantes. " + DISCLAIMER)



Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [19]:
tests = [
    "¿Cuál es la diferencia entre gripe y resfriado?",
    "Tengo diarrea y vómitos desde ayer, ¿qué hago?",
    "Me duele el pecho y me falta el aire",  # triage
    "Qué síntomas tiene el COVID-19"
]
for q in tests:
    print("Q:", q)
    print("A:", answer(q), "\n")

print("Asistente por teclado. Escribe 'salir' para terminar.\n")
while True:
    q = input("> ").strip()
    if q.lower() in {"salir","exit","quit",""}:
        break
    print(answer(q), "\n")


Q: ¿Cuál es la diferencia entre gripe y resfriado?
A: La gripe es una infección respiratoria por virus influenza con inicio brusco y fiebre alta,
dolor muscular intenso, cefalea y gran cansancio  El resfriado común suele tener inicio
gradual, congestión nasal, estornudos y dolor de garganta leve, con fiebre ausente o baja.

_Este bot ofrece información general y no sustituye la evaluación de un profesional. Si los síntomas son graves o empeoran, busca atención médica._ 

Q: Tengo diarrea y vómitos desde ayer, ¿qué hago?
A: Prevención de infecciones respiratorias: lavado de manos, ventilar espacios, cubrir tos y
estornudos, evitar contacto con personas enfermas y mantener vacunas al día.

_Este bot ofrece información general y no sustituye la evaluación de un profesional. Si los síntomas son graves o empeoran, busca atención médica._ 

Q: Me duele el pecho y me falta el aire
A: El dolor de garganta suele ser viral  Medidas: hidratación, analgésicos habituales si no hay
contraindicacione